# DMD3C++: full KITTI train → Google Drive

Generate **only DMD3C++ metric depth and confidence for all 85,898 official training samples**.
No DA3MONO/Metric3D/DSINE/teacher fusion/student training; no validation/test teachers.
DMD3C++ internally uses its official DA3METRIC-LARGE foundation, which is required by the model.

The pinned provider and 352×1216 preprocessing come from
`production/AnchorFlow_V11_3_Fixed6_FullKITTI/{teachers,full_data,extension_compat}.py`.
Outputs remain float32, with depth in metres and heuristic GT-agreement confidence.
GT is used for confidence only; it does not rescale the teacher prediction.

Select **A100 or H100 GPU** in Runtime → Change runtime type. L4 can be attempted,
but must pass the actual GPU smoke. T4 does not meet this provider's native BF16 contract.
Run cells in order. Drive mounting may require you to select and authorize your account.
No email/password/API key is stored in this notebook.

Each verified ZIP shard is saved immediately to Drive. If Colab is reclaimed, use the
same runtime type, output folder and notebook to resume at the first uncommitted shard.
Run only one session against a given export folder at a time.
One interrupted shard can be recomputed. A changed provider/environment/recipe is rejected.
Full generation may span multiple Colab sessions; closing the browser is not a guarantee
that Colab will keep a runtime alive.

Sources: [official DMD3C++](https://github.com/Sharpiless/DMD3Cpp),
[KITTI depth completion and dataset terms](https://www.cvlibs.net/datasets/kitti/eval_depth.php?benchmark=depth_completion).

In [ ]:
# @title 1. Runtime preflight (does not download the dataset)
from pathlib import Path
import os, shutil, subprocess, sys
import torch

assert sys.platform == 'linux' and Path('/content').exists(), 'Run this notebook in Google Colab.'
assert torch.cuda.is_available(), 'Select A100/H100 GPU before running.'
assert torch.cuda.get_device_capability()[0] >= 8, 'Native BF16 GPU required; T4 is unsupported.'
assert shutil.which('nvcc'), 'A CUDA development toolkit is required for the official BpOps extension.'
assert shutil.disk_usage('/content').free > 25*2**30, 'Need at least 25 GiB local working space.'
print({'gpu': torch.cuda.get_device_name(0), 'torch': str(torch.__version__),
       'cuda': torch.version.cuda, 'local_free_gib': round(shutil.disk_usage('/content').free/2**30, 1)})
subprocess.run(['nvcc', '--version'], check=True)

## Drive and storage

The uncompressed metric cache is about **274 GiB**. Teacher ZIP compression is lossless;
its final size depends on the predictions. Reserve **at least 500 GiB free on Drive** for
teachers, RGB/sparse/GT input shards, and the consolidated full-train ZIP. Drive quota
is separate from Colab disk. This notebook verifies writes and reports quota failures.

Colab only holds one input/teacher shard during generation. Final assembly needs local
space for the complete input ZIP plus one shard; it checks that requirement before starting.
Verified input shards are retained on Drive so final assembly can be resumed independently.

In [ ]:
# @title 2. Mount the destination Google Drive
from google.colab import drive
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive mount did not complete.'

In [ ]:
# @title 3. Export settings
DRIVE_OUTPUT = '/content/drive/MyDrive/GeoLift_Data/DMD3Cpp_FullKITTI_Train_v1' # @param {type:'string'}
LOCAL_ROOT = '/content/dmd3cpp_full_train' # @param {type:'string'}
SHARD_SAMPLES = 256 # @param {type:'integer'}
MIN_TEACHER_PIXELS = 1024 # @param {type:'integer'}

import json
cfg = {'drive_output': DRIVE_OUTPUT, 'local_root': LOCAL_ROOT,
       'shard_samples': SHARD_SAMPLES, 'min_teacher_pixels': MIN_TEACHER_PIXELS}
assert Path(DRIVE_OUTPUT).resolve().is_relative_to(Path('/content/drive/MyDrive').resolve())
assert Path(LOCAL_ROOT).resolve().is_relative_to(Path('/content').resolve())
assert not Path(LOCAL_ROOT).resolve().is_relative_to(Path('/content/drive').resolve())
assert 1 <= SHARD_SAMPLES <= 1024
assert MIN_TEACHER_PIXELS >= 1024
LOCAL = Path(LOCAL_ROOT)
LOCAL.mkdir(parents=True, exist_ok=True)
OUT = Path(DRIVE_OUTPUT)
OUT.mkdir(parents=True, exist_ok=True)
CONFIG = LOCAL/'export_config.json'
CONFIG.write_text(json.dumps(cfg, indent=2))
print('Drive destination:', OUT)
print('STOP file to pause between shards:', OUT/'STOP')

In [ ]:
# @title 4. Install teacher dependencies (keep Colab's CUDA torch/torchvision)
# Do this before importing the embedded provider. No torch replacement or student install.
packages = ['timm==1.0.26', 'torchdiffeq==0.2.5', 'huggingface-hub==0.36.0',
            'safetensors==0.6.2', 'einops==0.8.1', 'omegaconf==2.3.0',
            'addict==2.4.0', 'e3nn==0.5.8', 'ninja==1.11.1.4', 'fsspec[http]>=2025.3.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
import cv2, numpy as np
print('Dependencies ready:', {'numpy': np.__version__, 'opencv': cv2.__version__})

## Pinned source and CUDA extension

The next cell restores the exact repo provider files embedded in this notebook and
checks each SHA-256. It clones only the official DMD3C++ repository at the production
revision, then builds BpOps for the actual Colab GPU. The isolated compatibility copy
only replaces deprecated PyTorch tensor API calls; it preserves the official CUDA math.
The completion checkpoint and its internal foundation are pinned by the provider.

In [ ]:
# @title 5. Restore pinned source and build DMD3C++ BpOps
import base64, hashlib, zlib
EMBEDDED = {'utils.py': {'sha256': '41def98c2ea1be5b39182bc14da81773a8786245bbf6dbb9d1d9cb6741120834', 'zlib_base64': 'eNqNVUuP5DQQvvevKPpiZwkeGAFa9SoXxK7EBSGWhQNCljupTEwndrCdnukdzX+nbCfd6RlGIpf4UfVV1VcPb7fbj4Pq+xJG64La9wjBKeNVHbQ10GE/ovPvwFhocETToKlPQDc/On1EsA4cjhbq+0Zst9uNHiIMdMp3vd4v27+9Ncva+mUVcBhb3eOmdXaAUYWoAvPlL7TdbDYNtuA7dfvd9zwKFLsN0NdBtZgQ822RLu516JJqlhaWPObM7VkBykObtePXkuN1N5kDaAM6oOO9GvaN2kErHKqGv4U38M3Xt9/Ov6KEPWPFBSC5IaaxUQF5QsoeOAyTI95Ehw+NvkMfyLUchybugg4nflT9hDPWIn8dTSRMNNMw+ixcgidS5AFPvvrNpT2Oyqlgna84K1kJbMeKQlB2bIOcVi/tk/SgaxmxEz0lrB2JJ0TrhbzzqSBL5LkYDo12PG8WN/BB+yDtIW2zStuUEIaRsJYEk6aPaz46bPVDxQSDLzO0UQOhEG61spRxgjtd6E6JtV60TUpptMHuKeoUrzZ3FZtC+9XbF3mO35nMhcu2pKzHZFS3JVDt23tplKk+qN7PMZzLRLT95DtevEvG/cnUnM4oJmOJ5LMsXVIb9KpGTqGXcCGw1YYsrCJJBJNQISbTa3Pgg/aeIriQmLMVq/CSq+tqSRH1VjWer4o9aQR8CPwFKwuoVy3KKM2dtaGESP91/jNgvC3gJt8TsLf9Eece0y0Ng5DTp7102KtAo0AGu9K96Kx6xintEX6PSXjvnHW8ZZ9MdCmB7eAxmntiV400XuZA6xA/4/8uXfIz+ZhK1HOqDNM8ZxW+qDLScyd/nahVh7ObH5z9jIa0az0iDQ5l7rAhhyPIk4BPpKHg5/d/wL11h5ta1R3euMnEwsaamvQk5qhe68ElP3ZyNcrzpFjSTpQuAUoZy0/KYm6WNVlnvcdRrPOSU6K8HK3XD4R6nqnX5Q7r8TjG0RjHDjYJQLi73u45eyPGEyteUSTSmZTjKVEgJUulQjhj9Db4lINUPanzhQ/xNDY3Z4FmlWTF08KFmQZ0ula9RHPUzpqBglsYSU9Gfiri2BwwKJrEank9jvRm0fOV6yAf1cfbmSo/9aFKtbZbBHkq9BR2XEWPySHr6o5ma/ofdZSLOz0My2Gj2xb/oR35SqSULBZzQEOseYpkZe9PFudWfWR/VY9sNsp25JOQct5KWf43pwBsP+m+kTlpbPfssYgodxh+iDI/GYphUPHt5q+8Bk/risnebf4FfzSKaA=='}, 'full_data.py': {'sha256': 'b82fd8e1691309be265026b3465de210a56f54d7e448af5cdd71b52231c631e4', 'zlib_base64': 'eNqdWdty20YSfddXTJTaBaDA0G3tByrcKseSHZcTJyVr87AsFAoChhRCEIPFgJIYFf99T/cMrqQcJXoQgbl09/TldPfg8PDw/TrPhZrPsySLc6HLPKtFnMZlLasLUShx8/b67OTkRMRar1dlnalC+zR+H+dZGtO7qGWc3MlKxEkitQ4ODw8PslWpqlr8rlXRPCt9MK/USpRxfZdnt8IO/4rXZklyf9Y8FmC2AVNRlM1QrarkztDgx2BdZ7kOIETcELvEs5a1L77EqzKXlVnN65olca1WWRKRZL6YV1L+IX2RpbKos3rji0rGqZ3Ud/HZ6zcHB18+/vdKTIV7/vrMF6dnp2+8AwgagM/n9ermjnZo98Q7ODhI5VxkRV1lhc4S7dJJfZHEK1nF3uRA4A9aW0sNak9bfp+rSuRZIbGNNcF7vIClqOVj7XoBm4TWaNcSob+lhLCRbwiCHi0IyrjCKWAS15k4Xrs2m9NyMZ0K59eokkkdOeI7K9fs1dkk7Mh2Ms6wJQThogxIiZpOtXB5DqqR5dQRlkWJVXbPM/RDHAjaLKV77ot/mV2VrNdVIcrZxBeT8zCoVZ5pnNeqcZ7lkmipKrV6rJSqrQJoAEw7hR3YYxaq5skg0xFR6GusijMtxXuMflb1e7Uu0quqUlWPgK7jGmSZAD27A0mfHJpwJma+kjmc/15GtXJZsiDWUal09uh6vnBuN7XUWEtkQCvSGdxsoGVnVWcrGRX9Vc0QCBjnozl+MFJurXJW8VJGWZHKRzeZL+wZSQpfPKhq2agGczMnNSER0bQTkmtpld9LkrJbRLswyXToOVgt04xUUyEu9PSmIqPLRxgoUkt+tapRDxA2rUgRYGIc2+87twEUeLfr1LBA4eBscBanZ5hvxZd1SbGpxa2CYcEZPtRhEsKgihNGmrhIxeXPl+fvytLRMIyqFnEB1aYijzdqXQct0bnKU0DSlNUijo0c/YggVzGLyFnotN4wDMYUWJNRKsv6LooLbI9rmTo7pHkBKULjRDJ1LY9Frm5d5+i4rNTvhsjxoiInrKs1nrNVvJDRyezsPDw+Cspi4Xhd/JKWZ8yFAnIWdrxaxXfTeHO9nhoqkfliwSaQAFWEZC1dIyTQbHRmJsiQBUqLmgFFz169Dv3e21m4u6cvRBCnqcuj3mChxn4tW8LwrOEP6fhe5irdFAj9+IF0S5Ic04YCaDKgZk1oiO6L9x22e+zYsGvNiF+WnNZ1pnL+mmhY0GNHy/dJjvFnPG9IY0h6cUvDmLSCzianJ2FPaiMVn9F5RrwkpvS7l4rDcxGIANfoJ6gf66H0Okuxd+488cZtZDz3aUEAJlfb6Am7tsMt8F8CBgdbCdCyFBjAu/FmPc75hOde6mQ52BE9f0c3+Ds6elpOhknCZgh2+aWPrESg4zrQGDAH/wF4rmOcAQPmgccWtUPu7Xnb7VjsmTNH1pNVCbmAnjhFUykA9B+G7v2tuIQkqUR+AIClosweJcoOgqxP4lZCLAkoKyUOCEs1NVNWzCWcP5HBkDcVAJqLGOLkW3TPiiRfpzJa1D0Q3gMTQVyCUborJHwvE/8Qp1TRoR442XU8Pqo7d341p/j08ebmo+BcQzldFTVF6pcf34onZrUVT9n2+CmXhQUVbwttzvO1vuuJCLY9mLCZIBT/HIxSWgjH+fp6XVBSNKnauYbLGk/PZbyE40Gx9YOURZctmDZrvatQnS5bgbnUxpQWSOm9jw4MDFrmklMOhz4tsYiRKLIJzSAJqGKzUmvt9MsPWrsnrkdcXkSRPJkCHuexycRlMtjPUed4TVqxGaNXHtoSqV3fhhYdyAVVDleY1OEQ77xkuVPzLW3B05akXf3Xr+vOBwmrF/ANs17Uf1ZFE/TLtvLbCXSO3WGUY8TG+e5qG9vDDY0CxuhNQhEyWmrbg78R9X8epO/jXMtRIrcOuBOiX6uWfOMx/aqpoKIfUdeL+o4R4UwCh8Earu+adzgb4kk7od0xKIjEN9NuI/UloP5UzdiKoXFGEq3HcOvRnmIyyn47cYtsYcBi0jF4ap62Yl1k/1u3uGkMh+JkTuWReCq2AfwFaSMn5NHrW6BF4LS48pyEDchsgTLPrmHI2X4dcy7XED1B1WSg5Rh7xMdLbSXg7pN9HXC+iuF9pzCWVfNEUJYiPd63eem+4R5kiAn0dNtRX8ABha0cFNR7Uj+BOEOZi1Fnjn791byC63LPF1fi+sMPx/RAns6QcoGEo5FBZXqBVqGGYFr8dnoanDsjVhbraClIo82gzoW0rqMVBmbBKTW8J+FYQrJllGFV26XQOdsgme2q+z58uQKyolzXkU01X2ExDNK/xYotGsGikUlEaaZ/V0RvIkzP49g8HSmzjKNwIjiubZPDVwgu91228KJ2i8SnriWguwTEL42P4txvXH0sZeePvcuKhoNrMALAbTpAw8BriQ16VuJqe8ZcAazantE3IjSto1nd3n24O53h84ybhv1lFQtsIqtsvonuYhQIBh6NDHFVxZvRzUilcmnQ0FRxXfl25M6obgu5oGlZCBRcEnm9nwlXkgOUYJ3IhReD2wNTMh7zqplp8DtQfOY+wAJPv7UnGDQkTOMfEn6Ou/puUdv6hy+ATnunJVYqzeYZADKe11w4wgaICpJuewH1366zPKUbs1gU8gF1nKpVovLhPVBP+1wj9W8XOvlszP0V6ZrKEMGSLPV6JVaZZuxpROzLQcULpab7syBbkeO4KDWMED6Pfvz5+urtZfTul59+uSax2RHo/or8wFi5t+w/n9/9+Pbzh6vLwVENl0xzsUH2cJnKNw0VOj6vCdJ6U/IE6p41ov/0jfeCk38suMC0FfITEd/uOazxa+N7VEkQR55UVbbICiSTabOG5QoDTgCziW2zcZK42LiP3TBJ2m6mMHmkGDE0AnMR53reV5NaW94joGqqmnvWokg5BHwfhs0xTNNJCk/u63cqBwF6MTnJHQjvC8pYs8nk1WlIwQ9PLVXORfiULfb55uo6+unj56u319bWbOTohw/XZ0hkDXrpdV73S0gqyjil+E1VaK6AqUyN+K6Ya1pAbqFLpaV75osTut8IEoU5zwvmAEBE8PHZ69f7AKZFFoKVnvI4RbYmChayZjfyyVf+kJXSbmMKv+c+dBtITuVSJU2Mz8+I85sdsj098tDL9Efau/py0zmZ8cSpMETEv0Vw6qHusa/fUxLvl62k3dYjdxQJmR+Q8qTLVH1hBTvxZhRJ4Q4hZxXrpTMIU6tNE6nQaMRL9nJjJobyHqUZZrYniTVbwWVv+ES+0FtpDW3Wz05CcTQ1ujwNjxsT4fkCk6fd5Elv8iTs93EgnWmUGMjJ7hL2zHPXIxQBbegiFN+jezbvVCR173Yv1ic5OeJydgZBeROt+7O4bFDF3J73rkT2R6a1wKf9ul16w/ROi5GukzzWWtCHH8Yu12K4FY2SeRTRwaPIRR8890VXMPjNxYUeJG+rt2bOpBYuFwhuTSW+D1J/I7SydfaNvRGhQgWVHcXnbZai4ONIhVaOTQvUpWfIFhjR6MnKx8+tINO9sg9JUOlFzdTeAumgpxMU8lYlfSsa5VKR31Ib7AJkUF3XKpNZDPWWadi4jotEujwLSdF0yFEWMpcwU/NLHi6+FT/JRZxswAlAsYLNdJ0lwlRhFeU+bW7WgfS1UsGoQW/lnTHJcBTYVDWNi7pG5+PvCn0bGJsPbUX1Ut8wo6t2/qZnK+2EfpqvdfwSUU4d3c0b7Pbpw5a2N32oXaoscehKz9JyuBUr5tQ2JBK1Ks01X26c0Uv7HA227LmbjQ0ckcO4nYBu549GtC5pge9qFZcRSjg5dUgswIN6iMosWeZyfEvQ01lssj5plDLaEbeDhDLxoGixGDjZd1P6dZgxZYtR816EGd5wvgPGslO8UkW+4UMJBT3xl+JU0lUWfEdn9PmUG0m6B63kXKK0QG25UtUm2CHLzRp9zkzu4qKQOZnzj6x0ybTQlPfMsQz02a+U+5KYSRaWqMkw4IETmItJbxzCI3y8xusqvs3lD6aFdu03ZSvP4eHhFarpjY02VoMoEXWyVMld0/gwkHGhpu/In2+J1gVVyyAO5STrSqvKNx8uraJkyp/Qn8FiAM2C0jFTMt8VEVoSeZoZT5FlACVVPT3pKY5ds9nJL8PtFLpMgx/tCcww0TJXTV9ja5m+FC3dnkDoKEcSiVco3I6PR6N92gDUapc42l/+YGecAaVgCnus3MHhF7KgT2CqmppVH5p31wtWcbFGj0/HcludfNepBGVd+5W677wsIzfxcbGQbqc3v00LYxjZZDJPjcAz3n40Ou3E5efvTr3xTHjwf8Znck4='}, 'teachers.py': {'sha256': '9a80712193cb546abc7733bc0aea7429a728f6eda93dccb28664561aba1f2890', 'zlib_base64': 'eNq1Wvtv48YR/t1/xfYKhOSVokS9LNlR0IvPvhySe8DntGgFg1iRS4kxRbJc0rYSpH97v9nlS7LsS1HUh7MocmZ2dh7fzCz96tWrz1GSiIClYRj5EY9ZIbi/ETlLRPGQ5nfSYR9TVkoBClZsBMtFlhqSxWLN/R17++Ht6II95DzLRO68evXqJNpmaV4wuZP1ZbHLRPNFlqssT30h5UmYp1uW8WITRytWPf6MrzWpfz+sL5Nym+0YlyzJGqlp7m/2vjhJ4oRl4hdRmmAjoL7Sa5RFFMt6hSgQSREVO5vJDR9OpjYJF3nk89gTyX2Up8kWFJozLOPYC3jBa+4v7/95eXKCXXsXn95esgUzhjychYOQB1OxEqNh4IYjLvh8NBpwfzUIVu6pP5v5wjcU198v37/74eaLd335N8XsDmchOCfBbDqcQZvpqfo3mp+KsTvn/mAgBu7K3Wf+8sMbYhaT0+FwJlw+dien/ulsvPLnsyBw+dSdTMOxP1vNVsEkWE2nuD13T/l4PBqtxnM3nE/EOAyMkw+fPn5qNjIKZqOJ8F0+mQ2GwXTMid+dhDNwunw1Gc8G81UoxhVbtYNwPJ3MT2di6s6C1cz3T+f+bATGVTgcz6fj2eQUJpidTsB1eXP9/qLmGw/cgRjNw9F0NObjycqfDiaT0WjohtAwXJ1OB9B9PhbGyclJIEKW8CK6F55fBty0zk4YfiLcTWvf0wMnkh6/51HMV7EwLZbm3YdrUXiBuI98SOEZX0UxosC0loNb9i2baZH0k/MI0X5dIkq24jLP09w0bqqkCOP0ARnwrzLKhWS80op9f+VO2cXPb9+wd59/tpVWN2N9V2zLmFNEGla1kyrBvFz4USZMP1zbLE9jUe0qF0WZJ+w3Q4Jqy40z5trMIAJc0ge+IXIz+hpHsjApJC3c9EmsF1C24ZEBVXkxGhp2szH6MY4GO+iP3jetA+61SETOYVRPJw8Y9YVJiWt6XhjFwvOsQz7xWIhEwgqen26R8l9hdx6iYuMlfCvMJ6xOtjMsi5xPxlgsjK0ooLjBRAy3fUwTcbA24OYeSZ9jMdP4suF5hkWk7CvoyjKjFsUWCMs9YUYgsmLT48mu2ETJuv/2zYhCv/fTm+t3l8bhHv00EHDqfUTKYrEGJJ6V36TfgaQHEa03hTwU1sWOl2WC4hmRjd0P8eRZgUcsGiWFyAGxnib0wrRMAhXkXZU7+f7fCM9FrFM9TpO1J+E5yEKSLI89ud0TXRM8L1w5FIGU3FMRUGoqhYR8KQxkwbG/PIh+FYHXaBFBSC6RcUpmInjubWBk44mxsrKQtND1u+/tL/aP56wAxCTs3Q2V1YB9+vjTP1gIrNqIMkdCRz6DgiHVKV+8pBbksTSJd8aTQGy4sWqcIqlpMb7OhaCsPmdlIsuM6pkIFs7kvF1ZYxcYohXSXARHl99bbf9HK7ZJ8+jXNCl43AvjKGNd+7HKbK06tGbSk3e7jh6HW5JFSWXby9B96O5BQRyqcy/MAROjyfDRHbrTc0ZGiSPyRl85htGlkNi0xLaw/o/G7xUQAyADFPeRuUXmxl4UAIer8LWZzuY0hXpYVAp8KoCtUFp1B5tyvQYuhBwlZVM2PYxMeCY3KapN+pDQIi2H5KEgOEtz6ajSVPMoXQj9FC2aJ4caIydKpMgLc4BmpcjNRifLakXq8Ksxyhs5yJVaqp8LeNFLV78IHxtQi6joWL/An4s1fJDvaiEfAFA/IY3fvf9yc/0PxUgxRemDQq6A+8mOj5h00bGtrlRRvlBX9gsB1fnhMYqvB7MQ+MjFUsf52vlFIo9R/9SSTsfCxm1lp2ojRFg3DnTtkKbSNOvd9PckWjAEzFWg/piWtdTiVU0C6vxpUUXFy23DVQOMFTnbRnLLC39jaM3QZMOGe24yO14y922/1ELqbWV8R7QQ0ESPSWHS7uepSWqL6NaplgDwoa+wsHnnIFvzQlIFNisBhqXw6Q65W7NYL2/8Uz1NdNZmd2LHWuhl/oYna0BMYwnlDw/LFxQcsMRvd8vp2e0Zu9fL27hoNXCiQmylaf2ucgPki5u8FFa3hyKZ4p7HpuXovrFKfABQFO48hJ5/l5aFqdO8js9qa9wvSh4v2nnFUfQeGADo5tJYR4VhG70Lwyajq7S0UYLuexkHvuHRD5dv3hq3NkWQ1s0hRTOzcYFe4k9NZrxo07BpQgkGGm2biDpjv5EOv1f27Oidl8lz6gZRGNLdHjpaUbQ6q61WBtU2q0p9W1tMBRw2W8N0sHFUh4S6RkyqT4dL6kfNJHOqflRr9y/sG903yPBEPGZmD598JbXQ3rqw+qYz+YszmLzGgy1/jLbl1qSlHNeyrCNiX6v1lHDd61Be3Q+dVfp4FcWADFMR2KyHdtp0JzZzJxaVnnyL+7+KxRVH7apikXph6nKfyKj0/kNSfL27B7hMmJVO37GB3Yrvd/amCSAP4pzJfhjXQrR1IARG+Kb68i1zhwPwoHQPjtkF7vNjLiX7oBxYxZB2FfnVQx8TFZ5nShGHNjVandTeG7ra4ETs1MivGjNUjjwAMufFTpUnAC/rM6NusBvGo3lXt8mt/D+zH4C81KMzlFaOYOf3aRRIlmbVaI+IoeYlBxb4d3wt6JBCH1fYlfel3RHXdg2ocAy9CAOuRWGEbiAE8pa56FWmRqWIVVOiQTvHAAbEqc9GnI7ManMANZFJhsmAoImzzyijYKTBzGbYI3WpLCoko+ElFkpqi3CtwGcqfqfYf4WqT4NSg+/HiYn2Ra/1je+zT5n0VmUUo/RRgzJ0B12hbatflx1qoI4NSqr7r0YluzMM2Cp+vlbzDYjVkNOLeb4GnOoZYBN6qmeAso0EHd9XKlTO9uTqcR+a6mMA/dU0KJ4Na4/yr0oIVtykwYGIUPVJ1Hqqvl0Eph9LZDE6AevsyTaosuIJdQhfs8rZURscK6c/J4h49AaqfdazVwXIHYcc7KgDINDXtJ7sqj7G8NY53FjlP0CyPLatdhnngE9znNdrkZgnSxVpJV574GX5IK7IzpU057gXG5qXFvZQ+zzduaGFeMAMUikSbQEbNnuN2JJH1KHup1qOlwX6KVlUYYM9EMQu9MOVgll3aj3jS61auzlzb92Gh2cRbY/OSZ0PaVDG4oZg3HjSm4PQwJbx4bylZ2+qRyOw6wTYS/+tkiWXxwXdggmfDUfVJisIIIgjyIpoZKLuC4wSuIuWRduurwzQV4deHFfoxnxC0hYpaLhQpm+OXj/nwvtw5X1OZeFdC0Ajqqn3EbXTC2Hgfc5nZivkP77tT1YKA6imZCmyo65MB5Sm8VOEfnMHmZlKRo3fvZZR0hCBuC7jwgPuO7CY8RJM0UG4p2LBoBNiSf1TO+0cHNjsDT0HQNbGAcCjOg9r1bIISg5Oa/aj7Rhi1OWpU3Y6JqIDn4NJRIUMFaYmz5TVWiabbXnm1bPFwvCz0miqrUfnEJ0GXKd0rlp+1EXTaGteM6rRRaKMRh9kcuMgjWAOGDOh4wOfRhuIQPpBhqWKeCVc3X+af/Vm1OcStLfnbIVB6+7Jdn+7A6Zt03s0CSKMHtXUg7TB2HM4eiiOZvBoBRFK6VHu5RDvoHDN82ToqbbZnWkoGT+mTCIZExra8qL7okYNQ0/l7g8+e4+3gieNm/V0Zi7nA2c+nwzs+dQZDk9n9nzsjIbu6LYG7UVVNpfqWO3MVh/063ZfuCyCJ7JP585wNBvas4EzGU5P7dnQccezPyRbCf+rFhclIdpgxIKC9GpXBPNwnLadwnbJKeY7saQAi0JBPVga+Xpl3Kq1biv7vB5OJu021ADXodc3DljOMQ+3JD8ePO10ihdNCvY1tF59Hg3P20LevHmjJpLeGTjsbarn8ar0aEhufaxHUFq9crap60qv8a7Vr31hV9ux2V2rFHlIC1n23NvlwB7c6oGBAiYr8Vu9cDP3kClw1GsHgiN65VAfGmDWiGRIQwRqMiaQODatr+PTR5ojiacfJXpibMxQNTZ1eGs9jX3lOyNRMw41o1Dw3CjUFAtIODLP2o0312iGl7BJZYXuA2/L5d3e085g1MxrSGL/DsrZvlUP0J1D0Hp0lne7g6F5z5hE1Mx53zFX9KbN0QFYgY3KARR2rcG1pG8W7N8gqan1LC4xZlpkpcFw/PLxzXV1xq3rN9twlPBElspDBEHVgS2rTpEr58CTegu/ijyVXhzdNTvteIEi/36pNKIGxNXHtNUNfXShEKSlojBuD03o6bdkjK+8srtIVeEomgNntY5xfLKuzgZM0PY6C1MqwcF9lUdHY6o6+2im7Npy/8OcXXW9dIy7ILr/fu5WY0fdGvZGBt2Uuf/yII5ZMafT+OallHWsvr048rXvxuz2PZQe9/RAlybp8+PcH0L6VHUCyoRA8Y4JN/YD1MM9jVLLs+FtdeRf4+Sz75D6W/5ogr+zYSQp2Ex64I7tnPp3c/Naieu7Y+s1/tt7Dx/2H7aiHptiqKZIjRl0qIROk6AAGtt0sTw7AxTbqipkqX5lvCC69x9vLq+9i5+/f39hOZhAE5mlUphDe2C7QOtUQVBVdWoQ73fLGWlgmo+9R7jwwatLsjOeTWxnPJni12B6ay27VdfqHxAPh3Mbv8b0a3JIvDzSCrwwQP2B+Sndr26PtooHby31+ZpdSuHlfOeRJbpHbnV5uHJaOwoT0qoJyGjLHMxPcWK6tmu/JgdYyg0LKm02Bd3CoL8QoLdHho0UX9P75zwBklQL6qp5vFoSNqoKS390gOb3bmfsN/nPoXfLvL8F3Pu/as6+WzBndFjIAptqyPEuaz/7FGXtMpWj9PychXYoD28jeFSs17HbTsFIKd2wt4WS5ELOyt6uDh6FtRwVeWRVLAWjqm/qPWQoaxLroDyi/PNvtqvnC/dBYTD5X1bWa2diH68Bzw+J5sHxNu+trOqo+pikplmoDuB0EVF/IvL0L0T2DnSJ5tk/ijgoSor25D89+c5O'}, 'teacher_cache.py': {'sha256': '8eb781f31a97c61e38dbecdbaf8c72c1c2e6f71b7c0d358445ffff48f2f94c2f', 'zlib_base64': 'eNqtWXtz2zYS/9+fAvXNlWQqU7LTdm6cqh3HUVxPndiV1WtzHg0HIkEJNUWyBGhb9fi73+4CpEg9IjtTd6YRCWB3sY/fPri/v39acDU7UDwWTBdcpgdZmixYnGRcvz5iH68+sZCHM8G+weVU8VDLLOUJu/71QmrB8iK7EylPQ+Hv7+/vyXmeFZplai8usjkLs1SLB53ICbMrYZIpmU6rjWpWapnUT38lQPN19ajFPI9lIgytnOtZg9AVPFYb03KeLxhXLM3N3rhMkiDimle7r8//N+gwuFQUyDQSDx1WCPit+Dyv6KMgqtrPdTaXYfCnytIOiwsh/hYdJiORaqkX9rBZVDN+9N33e3t7kYiNqgKU1A3jKezLEjinZOQd7zH409mtSFm/puTi0s3x0ZhWC6HLIqWb4fEbRwukVwRFlmln7LEuEYR/DJ0uw/NgGcdP84XjVUJkaSrITEshLH8kBOx3caj3+vPbSBZuzguQV/VHRQnXEQ9S6SC7pUeP9kYToGqt51v+LjHrMqcQYVZEyrfrTodpORdZqfvf96rjvngQYamF61wNT84+nLA/M1AFT4J5Fon+7ycXzvrO0+HgZDRgo5O3FwN2/p59vByxwR/n16NrZlmSekaDP0bsanj+4WT4if0y+AR2TPNSB2A4WkNrhjIX9mH5erLQQrHzj6PB2WDoWQGsjaKJVfYdTyQ4mgh4UfCFcnlL3zJm3AeKQP2rPnOPOuwVeqLHsgIWIr0wC2nuVwEHCymoDd5IFcsUFOZyz+dJ4lqSJASXSrBhmaIiB0WRFW7snKckC3tE/k/MWpYR9y6x6hqCKHMp1DF7tLI9dR+tME/2lij3zeHYn8vU9dgPrEcC0xv+AG9+ZIeflcYZWe7gCzE6eygYGBzMIdhNr3M4XvIhj+73mTMXupChw3gakQpcl9/0xsDKP/TY18w8/cAOj3oeKCRd7FCI84Ho1XqYITxkxl7M8GKRyPVsgyiFSLiWd8JZclBlTsDQJzWAVL16CU7KVLt2h6/Kueuh0g57R98avfXGN3Z17CsdkUoPxcG3S/LbLjG0ktTXkAp1qjRPdRevJFNVxrEMJQQo3C89ULeLStgaEfJCRDLUEOX3TVyCx05F2CrzX2x49takgEoJwCIWBZkwFXcgAoKfYmejrgJcAImN5/vP0WKVBO6O6ldgCg5aBVlunGI6ccZvCOlbKIVIroRuoBSeunFwozNuGsKgsYsLHkaW2WfeOuNn6PvsLQtnPJ2KCGAb72x5M0ob1lfMVfgU7ol38eUcdeIqXRjOXss3aCPY7WOWihdYnFAKzQFuGgIKsphDLowaIoC6rADhnT7NEjiOD4VQ8m/hEt8OZb6b4+ODw7HXob2nlxeXw+Dt2fAIaDeIGWCz7uBbl3GBh9ljUiVaapk4XfKgDSZCjA2TMhLBVNusAZ4j40UAPjtrJI4tXA35yn2rJKXmkPbWchqUHb+WENXxgr29HP1MFQngTKFYlqKwidXkRMRZIZiQGuMIKwQ2FakoOGZKBhFVaEU1TMNR0S6t56wIZ/SCSgYrWF01TEqZRIF9W4dWYPKLTar3oMJlHWJu41Dl5XgAFEbXKAx4YJ8E8OcZ4GGWytD13tRA0G9za6jFVhogqR+WEWD8Ig1nBZwHpwACxFyBS6UQxesc2EHFfqcoJvBh6Tn4skOqGmSeL9pa6qV/mkp4oc9udlGRyKmcEBnLghIAPPnfbUxgIoGg9l9/562c/xqOGBlunKkO5lzdOmMwuU/lK1wPiPQo1VW7tmwhsmEGsIGlJKSeioXNPRUSmy0/4GX9qdCuA+m88pcglw8gKBRjmKZ21Ra2pqBoOrv6jVEksrX888u7KvVAcUHcn+qyyWbPRwBtpQDGjpnBBQdJw5NxGcdcPZARvYKcADUD2KYFm4xqSggp2NOOsUYUAK1KLdVdj41G1oiRV8Iq4rfx0CAAX1CAC0GAhKZ5iayWzgvaDCJxJ8FdUz4Xbs9bI9oMMzjcfASKa+4OW9berdHMBb+FmypR3IkomMtJWyqozYK5mGfFot7kYrI8evXqqPdExBp9jbvMr/dZcWvyqkveDB1FAHc2gOvjZsfrWBsae+YF+t26YwDE3vMi6kKYgo6sn1ydXF+/aThHv/INaK2Ssg62qnuZz6UOTAW/hibGzB1WxXo0sa67GwysSP0tTdrS2ZrbfdP6PLcPiqMOda1ZwYsFIpjtYOG8wt8uIGUsH/qOb/32AHQAlPstblaAYrGMynvIWtBV+3GU5SJ1kY9zP3E8bHrjdlEBrYPid8KNl1qC9iG7hygIbxPRf88Bn7zWkdgnOyAOIw+EZhfegeBp5jaqGVgEJ0h4KNz6lh2rqeWuZo92/vF6MByxyyEbDq4uTk4H2E9dVv0Z++/JxW+Da/enjv3Pc1Z9nv7cBhIYQ0EnMxUFeSG9s25hyz8rUSUa1Nxcux78E2Bl5LVENf4GcQK171U5SaSasZP3o8GQcfRFACMNhMFW4S2g60HlZ5FRrnECW/yCUKDphtUowmpNeX6ZJjK9dedS4Qxk6TrG83mCaWoRRFAnbnX82uGNCsHHmuq+HlwMTkfLBrdjj+FP086+H15+qNX/+8+D4QDHE/2fwA+bWoYWKxY6nKEoNuJNUf684MG0aORbq3ttxUde2N6LcxBqk7dbeFeeOqVRldnbNWUfdqAJFCcITdgb39/sg6D746c30M7c2+EWjSoK8Vcpi7rCBrmwDcWrQi8eYDS41LjTm7ZLodj2Eq/HuGe1D7GLR+NdFyBvRPcygsGposyxQoWMitRAbAhBLgvwvBzuJXUCIWgaM4UBC89M3crc3mIVFwEdMBWRZBuBoQmZ1lbopNZHFaQbaCuCKZCsHOBePXO69OZFM6XKHEDR9NpoQEpPMmVuVW11Gn1mwztsiOFUCkoitOMWx63ctmFi4oNM6GoNx7XyfNOvyb9i7hH8j3qs3rj6dYi/voUsenj0n5W7fGMKtroiW5YvlK8DqFfgSkdQBr7CtP3aFns4gcS70LTUj6S6DUoFdqAZG4QqLC9jybL6kU7t8LdqTmPtCo4kABceKyJdkuHYP4yf2Jl8SxQhaUP2jtgjPjQ2+MzZhN118fK+MVIGHeAANCO8ZPwuk8CVmiDUKva7c8iUCiswn/2GMwaWIJRD8GFdpvA0u75+57enco8OykRKPGZL8chHzIXsWvt+T9a5bTvYAl9QTG796stbPvWZnq8Cc5xA9reXs41tNLlszJAtNNaOAg3VM0bItoQ21V2Nr8tp7sYpsilM6g4UkX29VsH4kQQMGEQCWhejVVQEtBteu1yRBCq5jyYRCpEPjm+sTZ3r0eWV4/km5bordDbmlgYTxIBnZFjKmsssu0KkMYVbn+ZUf8/s0Jt/L++km3/PKZnNvRpl86p+JPs3tII96jDRBBJ/QT5xGwDfEnljJ7Hd3fIim2JIW4f7HFRAHFMrLqlNlBgrmeZJYCLmeClVp/LiwE74jhu3fVpX1ErXElbJtluxY4/yqftY01/tUDYWeVhDJpmq6qT1nMnLSBLkfw5HXg4cWAioMqGm+skOb0/skBCWDogtm5dKk+8nApoCxiEgeZFIhPwyDMEecZkwzOYgpp5lhdSUS/1ndYs0QaFaoIKRRntPUYgdPVAssb91SCIAeMea5iUJvUbIBuzZb3AvRrv2fNZS/qpBeisKezvnt0vnapSi0OLFMRrbfBQtCyx9WC4BYaN6XtkY7IYZOKQZL9+MG28pa7ZfUmNoP6q6GzGbesRo0pYcikfYSMXVY1UDtQufp1WAgH29xp4NbQd+bWv2F473hGqteG3A6536I/cldbDBHyeno4tPWLOaUoGdvwMoM990utDbaCq8y4kS2lnpb1+UkV6WMLYki61fpWzlaF3sjb3KJMnC29Zwv/7w9E/MMFvAzeiDI3UBO7u53bODtRHpy6ajG8n8s5PSdqr8/NR0xQe+aIK608PPV7/ZnY269OlOpMo0qRgAhvxxq2ddHau2r2ZAw+d5LlKwLW7ctI1QpNq1KYyhvPySMQFof6MSbWVhSovNalpJzPRxxgrarb68mny2O0GbbIGZ8QapjWnoTI4ebawg7BKACCz0li8QUejN1lKFvGF9yIwfzStjIIM5lHR800b65o8dudmxPOR9hufmggfFNvpartTpzK54dZWQqAzP4UyL4HQmkuggKzWrRya4X1WfzKAfg+4LUQvaMMIpTFo6C7PErxO5yhOpTSZHVWKdggr02n2BheDVmoYOrybXZ+GdYQtZxrDb/p1x6/cHS5FUuV5R2GKnWzNG5RQ81PVYvCo6FFFAv3v6wuKpOWC3ZaR5tfd/DZDGrg=='}, 'extension_compat.py': {'sha256': '8aa6d2f492551a3cb05ff84a3673745c556a22af74324bd7dfbd73010461d6d4', 'zlib_base64': 'eNp9VNuO0zAQfc9XDOXBqZTNbtML3YUicVkB4rIIlidAkWtPGtPEtmxHuwXx79hxuqVohR+ieGY8c3zmjEej0XtkNZWC0QYcSqvMybOPb4CpVlMn1qIRbpfBh6troBACNwhOwXN9pS201NWnWzQSG2iU0jYfjUaJaLUyDmpq60as91tbd040SWVUCz5zcMHg+ui30RFC7N5MnWoFK39YJTN/nBbzRZK8vfz04fJd+fn1sxWpqgXnZzO+WFbFrJgsl5SfLeZsiZNiPqWUTyasmp+xWYGPzqsCz+e04DNcnLPperaePVqTJEk4VtAq7u8gfmLp8Nal4TO+SMAvUe2vkUcAvTNHyfyRdDzOa7zlYoPWpeMHqwO2eDosQ4VF+NRJJ1q8NEaZlFxVlWDC8x1ZHAjkoqrQWIgMCSmRg1WdYfgYDFadFXIDtPOk+L6wwCGryXgPs8fFlK+TktzttEdHPKQpKHPs49TRJ8G1/D/ILxJvNTLnUciuXaMBVQFHbZDRYIxagaAVKxzaAYpB1xkZSxrUDWV4AJQBya0XGjXlHuJfQRFZiAl/pXbG4xxa5MtqarBcd6LhqVFq36HI0CpIKJpPycv3L6cvtCanxIOwpA9z1GzQHYX15MeEpVOG1cXkjDyOgXm75cKkoaR0dnVtOszwVlhXqm2/i5etPAGStghCQkqsv7rO9Y5kZK1L1nGaMw/jsKsP/2Vses46Mj70Ic6Ib5XeFWm82WnIn0VU/X+s/BCed80W2sPshkYYvDG+FwEOlSCsavpW9XeEkDWHO+29Es7PM7Kt6vxwOrqzwBqkMu/zR3iroe49qCOrnuDVYDNIeZyffjq8WFekc9XJchDGENYDjHH3jV327+FM4k0jJK7INzlk+utdSPcAj56rPLhI9osoIzZC0mbAXcYZJheHQc3uuL9bfS7RIP/31PACROv43pP9+2jJxVcyhet+QAbpw8lTOFa+1/lyH9Mr//ppDLsTfzCQ7/fVGYCF57fsZKzKyUUQ5u/jMezpSf4AyAXqSQ=='}, 'bootstrap_teachers.py': {'sha256': 'd4c438d1d44f66a266cb8ed5890fad548e4564ca286e09818a3117149517ec76', 'zlib_base64': 'eNqVVU1v2zgQvftXcL0HyogsY9GbAx1SO2j3kCZo2lMREDQ1slhLJJccJTUW/e87pCSn3bjYrA+JyJl58/04n8//NAFl2zKnjYGKIUjVgGfB9l4Bu2C7XrcVe+tuXWDaMGyA3V/dXDO0XjWrzeftFQPzqL01HRgs5vP5THfOemTS7530AaZz6HfOWwUhnG6OYVZ72zEnsWn1jo33d3QcBD3qNkzXHmQlvgZrBtkY6km8vdmKze32Omc3tx9u0+dsNqugZqq1BrLet7mHRx20NTlSdICL9YzRT9fMWGTDXQHfdMCQjbL4GwXdodI+o5wo0VB+8j0sTirPyRW+N9kXvtfIc65N/BfQZ6PDh5xiVofXWS83P9nm3ENnEUgiq4r+Wq/32vCcMnuJOyY1Gq94ETEXhQ4iZrFYe6kDsI+9Qd3BtffWZzX/QBaSvdPIEpztcc3+HhC+8wGXrPFY/hBx0hSk6nr8deg0ZdiHKFlStxS0MkZOJwrAS3WAalnrFkJpLH/IEb7hkEtBKNplp5yS+/PBfxpndwqdNTKw1irZss5WutZKIvU+PKd0SSWisQpUH/sI/slrqu7g6rUtqQFVkxKpwGH8+uOHzkzz9rI9r8WfkhldoIzefg1LFfrfrSG0ZdpUEr6/vtqerf9v5eT0eS/OdOHUhEmbdTp0MtZoMW5jR52ftsuVE0sUV37fRwq5iyca0EtX0JQLOV5nlL6yptZ7nuPRQRlJIic82beYDpkQcYKEWBRPGhthZAcZH2yKSBt8MdRI1fvyxCWZK5L/6IiWvhj0F5fe2hGW1L9wbLSvBGniUUQRfxixErXwBtGF9WpFFW76HYF0q/tGehcnOqyImd5snEsbmJ9oKsKs+Cjj/wX39oiwlUbB8h6gWm3jsC2vzJECM/vlmwH7RHwT+L+0Ri+/s/c2INF5ov5hKZghWJb4HK1tD0QBmXlUapGzzcUFoxBiMj5nRpuvMmeppwQ6mdBrUCTwxM00PmBi+0U0lDhxtPMQ+VOkVyWpp6/yp/ssBn9+S+jFIH4G1aPctUCsAthTA480uclUkOO0Kdq4ViqgUVZPVZlkL3bFeR3n6u7syyfN9OrFUTkW7CMQj7y7+8xCZw/AAr1MLcn+6rWHquB53fahGdFnM9pDkSZQiLLkQsShF4Kvh+Gf/QNqqoVd'}, 'colab_dmd3cpp_full_train.py': {'sha256': '39007266a5c3d33394ddcc1b399adc02057f00adcef6770002ea1892107b7172', 'zlib_base64': 'eNqtPGtz20aS3/UrZp26A2hToCTHvqw23CtZoh1VbEcly6lkdSwUCAwpRCCAxQCSaBX/+3X3PDB4kKKS9W5sAvPq6el39+DFixdXRRCn+1marNjZp7PXp69eMf6QZ0U5ZJWI0wUrbzjL4zTlEfv18NB7zfIii6qwjLOUlTwIb3jh7e2dZkkwY2lW8lmW3TK+nPFIwNhYsHmccBakEYtLYQ+OeM7TiKdhzIXHPmcsLDg8lnGQiOFewZOgjO+4XmPIRFlhM1tmEU+G7C5I4iigmbIC5s/S1TKrxH7JRckWPOUFNXp7L1682JsX2ZKFWVryhzKJZyxe4hbZ5CEuv5RBeDtkYZLhdnXPJOEEpdBdIz4PqqSM4rDcU69uAnEDk+nHONO//hBZqn9nQk6ZB+WNtfIFPA7ZRVXwC1j3AR/1CHFTlXFinv6dxCV/rR/LeMn1729xjrjd25v8djE5vZqc+VeXJ+ef2Zj98OaHv/+w9+7kywQenJuyzMXxaCRee7zaDwGHRZDsH3rBMviWpcG98MJsOQruFvu3cVnGzt77899gsk9X559w/OHbA/Vnb28P0IDr+rMV4NkNivAGzmjI0mAJf8NxBIPjPQZ/4nSewVgFovevOD+HN67px33cydg9Ojg6GLK/D9nha/jvaMiO3g7Z9weDgZkFgcsLLoRfrnIOc6pFzXs4Jeqs398XMdJA4eJoBZSCPIoX0OLiWSg4b2BCdY6euAmO3rx15dL3cXlDpyR7exkQq+sUM2fAAqBpORr/zIH6ZkkW3gK0QOK8cJNgOYuCYzb3Ch5E7g/sJTt6+fLoYDBkM8cZ1EMJAK/KER8uzSHXLnhZFSk03fAHBbLeQUwcUq5cIP+Kq7l0/+Y+kAq9qFrmQnYGDgKq8W/5SoyvCnrmeQBckhVi7DpDZ8icY2cw8IAlgcVc+NVdH3fk48w2EtX6tGCSBZFwLcTRCOQ7mFBNIoI7Xk9CrGz2gm/gTOoJzFsPgIXNe8vbKC5c+aB3wh9iUfrZLT3qIQWKEpiLBuN5+kh+NKuHv9gr5niqm2Odunqljvy+c+IGsRqv8yGcPR7MGAg4SJLsHlZKx+9BkClgiFC8eVKJG7d+kwlvLlZp6EITcEmauYruoaHgeRKECC0BM2QSGRKB84Lzb7wHe/HcplnCinAHJH1bJ8f+NpYj632BJhCcXVYpsuakKLLCnTuXPIxzPhJZVYQcReg8ARF4zB5xlrXHvsKQgKX8XqkN4IckApWgENp/1IYdxa2/QOIvsgwUziKeqX3gBuHkpCj0qF8lgoXsOPCoeYRc9fpA75ve/YhzPLGjj1kIZIFzHgPcoNYecYx3OF+zD/E7mgeONKsAZ4/4QE2OhjmvZkkMp5jgLCBdgDnilDSNAh3UzWm2XMYg+lGpBnMQCYB8JCZUp3gUyKhatZ4VILP2BfA1IDdfeaitcJru/JorqAVkCT3Y69M468UzGabgYVZEsMqjg9ToHDfmkpLbIbkPTQSEJ8oA2Br+8UX8DZul6MGhUmxIWNf6jOwJNXHWp9Xq0JycjccKwmsFxJRwqRay8WB3VQBNm0JXCSzZaTu1TBBMPKyQDrXk0T7JDmTLeRCCYRDP57wQwBEWDGtF/iVHVR0UK0CrvbdaGrVxvEkoSYQ3tZAowmG9hJZXqjESZb1pxUhIYXi22ewPlwZDpyGrFZTpD+83SytqbMgrOLkajNa5/a17bqAx1bGZUQO7X/fQeo+GWIe4ht3xIp7HoeSTeQDARXAgZnZ9HJZcNW1NHt7rksfeXpgEQrDLIF1wMGQMm1+CRGXZHNZFPfOv8wumbCvYHYwFvQqWrZK9/67gVcQuPn9gSzSOC8EyAJr9dHV1AbuDmYVhfRQzvh+ncen7ruAJqJeqSCxGwXcevAKigr/3rEGwPi/UKJuzpME4FyLnYfstAgcYEDWyucjB8kUBrNu8BS9dveyQ3cDOYQvjR4eQAvwuz3Z8sH+AZsRJGPK83J+gIQG8g+3acHHWwwYr9v0B440HSyWp8MSzqhwf2vRZFqs2R0uYPaIUH4wyH+mwEhYBK0o1XWUHH40dpL6jg7dImc4puglpuS+3hj4NWnZmlNr7cWcXPTTq/KKpQ/ACjzvKgCJwRlHlEvm4SETEgwpa/ANa2bxKkn1lzgI1J8kMXBSnuRE6DKWVUWTDuQCa6yMykjpOS7cN/HVrk1NP5OBmuM7IGVzvH04HW8/I4WWAZ9qelYjEmVxBoxL5pMqBtZJk03Gh28WtM6IN3ADPJLgtSbBSsNVbQwdwjEJwKA1vkjNjkmDgx6G3SK7C2KFWeuE8TXRqZJaT2wekvQweaAJE4g9r5QG0IP3W9HHew7+utYXBXkvjYFuDXx8sJn8ZFAvRZvNv21Bk2syUYGcXoJXUjFLUDJk0r7RLPf6cpdxap4R1eQk7kWY7mVhgXbnGBQeukDMN0JwXWXJnwwMshRQtZ/Fi4etxfplZM9YjWx6Q5Jtf0TDUkv1rKoI5J66QC4Mwlz/WFhsoH1OhCRbHF66C1AaPPEEwOEA2Ai36QVkW7J//BMd2wP6bHWSH/4POLVoN8PsIfz8BoIOAidUyidNb9P9mcQTSzYJM4eJ59pdR9WpDRG8IudT1xOpDPXVL13Nb2W9S+GoG7DxUCp+x7xjQ7eShRNLVMRWQUaeXp56t+SsUaq5cfMhcK0IwZNbDoEPxcogynnlaLTEuA6SBYSd3UfrAPeCIAnJAasNvRRtoCQk8W5CRPHJT8rJTlMNyCKAHu4AlXrYEvEWUKOCLUiBOXYcWHDnSF3JGeZH94UegpG5GiwJt/bKo4He8BCfDP3BwpZS6gqGaRmoOLwddVps8YAFRQEOQ8dIKwgC8CUmtUvXp6fSEffMe1EBtX9AGtCUh2GNzqjWr0hh0NftwxWgToEmUD1ODqZlHodsgGXW7OYEGamV/xFDOI7JhTRTMxR5qvjjS88gXdFpozCIacZV6pzm6MXbYi2CzLM+CAkphsETPnSxhcb3/ZjqsH46mNnNjT6WjXUed3xEaIer3a+cpeQMzWNKVEAFrI1jGWnQaVIKzj+54kkWrlPtFcD9yGvJGzaGgsrHdB0nr1D/FguKueUAWo5orTvMKvW75aEtBQD5qSueRMLf25bYfc6B+vlz7j7C7tdOADgYgWHBoO4BzVoFlAIY1B1d+mSccYYijFgDCC6LIhR/1W0Uz1wTV1AtyDPa6jw50QjsljgCH1IbOojxzB0AF6QAv4Ac8Lkr4qfxOuW8cST+eUudOsZj5UhPAGI2c6+PDg+l6pDA1ItSMMD44AnThQmtnLXcAgtCnXkjXjy16HdRU+Vy51BFLsNCTQmmrNDK+NXKfQjqqtXoHW6VMbaFKAWlF1Wk0eSlJkDsNz4j4Xy+mAyNJkPrguRWRkE0CuPimSm/r0BQ9sh/ZIUpH+QBa+ODo+zaItq6lKX1JfIItK1GyGYf/l/ecpzAVogXnUBAiGHBm11MjiBR1Fdm9ILqXKkWC6MXAJKJhkuAQOiGy+tE8dsGsRCGKMwyamzJSDVatafyWr4jqkEz2Sf5i++D44A0wTg/dq82hSQ1LXNPqx/T3K1psum5gHydTSAeLFexfE5pQcTRcF809jNzhkShgl0FxCz6ICiNpM8+Ro4WDJh+OvCb4pxSIwMidYxQeCjQ5SU/0RgGHtiW9k9OSD6kDj3KsmU72uHZqQMnrV5EA9YpIRXWUePJB3she1+DIoDyRbEinKzegETp9KoTwBYkLvRGwS5rBTZjmBeDhxVSLOlziNk4jpWhQHgs8TJWfEk4DFyqUpiDHcbXSskPc+hRo5pGJgFAAbto2sSlybYK6AA+9eE68xYR+n4iP9eHqVFOZjFkyYkzYYVFUeTlaSrWlA8NNeSHRoGh2GdyC0gRCdyWdAvwL4IJFqe1AIN3gHk2AxIoHZ0tyi30U1zpyAa5tEaciDkHUIMZ8uauhJDlJBBIM5H0jFNDCpbQAkorkxSmpBkm36nhQaUDH9Yjg8EFXgD+D/3jlQykVKrV42uUKZYRbbcchswCYKKyzGEhCKHB6CLXG/6I0M2Lfa1SDUzVt20xp9VRKstsbYGl1tfTjtAtzt481V3ZPAQepyNXCyISWYJNvFWo7utr5WYYl1Nm5tDpKIlgepZBCpJxE2QQ9IYmXLx+LDA0S6+jNXNJskGggvGPXoTFK3T77wcUdO9vRAL4PHoiGDs+G3inU6/f6JAaD9drGHHI2aJQih73DUECkSehBq+08GQLGBnNCcRomVcT9Ran8SAp8rnzMKLRcSaR5rZLM3DqyCW02NyoZJtU36TmzYlaVIOmGdRq+Fs1DtozTeFkt/Tx+4IkOXSjmBFcvX6Fzmub26zIrwpstPN1mXeql4aMQje6pnVU/KIpgJZTSobCWxeykRsl3QVnmLTMQpFkah3aatx3B0Xt27uFArTTz2HQ8v/DPJu8/noD/VfdIOPgE40OVAoTJ3n4vzwSx8K1pYcRkkCApGp9Y4f2wZVpIXGg9+tdoQqbJEVuID1W8AbCTPSenbnZu4diV/wBmlhzYN2zFJMHuXcQzAlb1vD6csn+OmfcGTVK1AnKNvwzELUjdg6lHdOJS0uag0WtDl8aKIVjJaGNgkFOv7oEydwedsK/s+mOLZncK486d81RUZChjAYpCHPv5TIdxQeshc78AOQgGAzzRYusWeqQQIdUi0Td6tLlfDR9cHx+hk2JN6KX5ymlMNasw6YQbz7x3qOXPf2n5Gyk4gMEdd2XPIdMnJ9PTeRzeJrydoZbGyQpT+DC1HIlxNUratuavi0C+1XFGNbh9RpIn2z4giUOtOVDkwzs9kbSvfJNO7IrOJzzAOhPZqorQINq1DXZeE4111eWpJTTBKVJCvxWPHRVIw6I9tmTmukOXMfsvdvSGaBsNtRh/GW+jS560faBIZsq1HmOgFjNi/Q9gwyAXmNduS7x9JQwpo43GK6X4WjKicbCOVKtCOgJDZtWU6GNVnPYdO091RZeU4vszKqw6vTwdss8Xv7OIU6YF/TX+gJnT8zPpvwjw5kAmcnRBU4E095RobktUQONjaPsCIUpWDeEaDd7Hwm4vyLkDfK2fNHydK8Xu0t4FoMHUXQZleGOxN80pjX5r4eMNnPVNFgZpG5xof9ohjB0I92lLfrdNKYDVcVTLvi32qQMQMgiTa0khwzx9kqatOMgY8MIqCjzMLN8UQKfftKQxvoNEpm2vGOnQb61opyFOZPwmVxbgvT51es7JECTJgnbfVB/EUEobNO1ar7RVZ45sXa+jy7hyz05/SAPUC4SfY/zIHVgrU9uIjN6Bt0iymeu87Hc2Bn/VVvly9cslWio7GCbaQlZhEtpbSwx9kyV2agPS0m4GcZsyhCJLPlZIPfTIEZJag+dy/EwxElb6wYhmOgoad/Bnz5GIKMckCwZQUoHqhsF1tY+qI+VuOF/Yrqh2+rUt+omo+krbyNpelXK/3/zN8NSoqkPHAmAN5TT5ctPkW9RN1NnHiIGWFtBt1+ySrB/ZsXNdmKXKkA7fNIJAFt/Gwg/uAH8BKEIZlLAawXbwI34XhxxoOQ9mcYK2zgBsOjDEftgemPzCsQSXnRweHIx+wr8+fs9SmY989/7wLTv9enbisavvWSxYlSpDDE5xtrJLlrWRa+pxlCMxX5ATW97EReSjL76SiD3G4gO565Hd6qDsuplLD6TZy7xV4kAeuvSR4fUSA9qHdaAInprk4VpAWQKybX04sr7OF2ER55ZppHSB75Mb7COc0BNQxyOZZYM+zVxRZ+ZGiBXNGKS25ku00cByNp5ibfJg356WjoGm94+1GaBbsCaE0qgUMSYpD7BS4sOUiO8nwQpIHMyULChfHzH3aPj6zdHw8Agzt2BSdLL6ymxcIhn5AhRjGiGMjWylk5UIhQnbHYOnCG8xXo7CxG4gpbU2BYnfSBiNlHWnRZl8GlhHT2FL27O3emjfYVwHSknqmUp0V2Z78acdE8L0Gr7zZIWPKmV3dU2Si7XerxxKZsgMgh+kwKnAwpEH8tEZ9GS4njmhTni155ORdJivm93VEb26swrP9yYLeqjOsrCsA6Bou0L/45PUzigOj+eJ454uBHFkopwoY1F6Om2uIm7ycbqrkV/CGSR+BGcNuz5oaFqZX2lnJNqJBQQdoxl5bTkQDkES5ybEhj6DTLQ1B4M1jhxlgiH28rlKcdAKXaPRhO/bKQbSW50Mw15PpktNAfIZaf24F/EKRO0b4rzduUCebBhuoffVWDlvjbhqx6qmOLpctG/TwLFpxRsNKlxcZxAbrbIMT3EL2HIUzBo5r6g72ILL7I5T7ODBdXw0cp0BdFTtrxxipsaEJA4MG2LhHwoEioi3mdNeDoB85fgyIk0MiqPosbtNm5c0sY9cDZHM+SBnoWl6TMFjzQQOzUgZWnJN6PW6e2INAtuEbV2dQ3CAiXrhDHrySp3iQFNRLuGnIr5aGuAzFSSKMgOaijCea4jXiIiabnYoDOt3/rdLnfVg47zSr+lt3mh3dSskMcw9ZsoEoUdnZGXvNiGbeu6A5UAIjrePqLup3GpXd7mN9e0Sr83HJwuTimUJROi28hSdLfYYrBt7bwvz/lkiAlsk5JTYcpQeQTVSo/lPUM92avyzhKUyXM/LrW0iEm2kbJXcLf8djh6M/ZwHt5gsyYoV1d8KdzMl1KZQw3uyzeHNg79jn4I0wrtLKww2JfuUQ/hw8ZWJZXbLTUQJPIF5XIiSEg3S2PQ2TrpbRmLjyJ2zFL0UiWD7svBgRyLeGs1vZQsE+FyGHDoJgs3zPy8BsIE2urGdHXhykVc+IaVmyxylEoaOpeNa1wPoyomDqY4o78CXuAIxX5+7Sncz8MaeQ+3K45N9fTCFC4yxoLe1y0JxOucgwELbJ+nGZmsC2GlSYjXkuuIOhMWStLK1l2XwoBlRd3IHI6r63CBCpO9E9ZiakuFB+gkYqJIlD2RcDPU77Sx1zRiT9NgarzNrbpBqMvqsBduWNKUF8YZc5UZHtX/p75gqkVe1qAkvGZrd6C1hwQPomio1olxeC1P3T8KM9OuGacm+C2hrS9g8jEWphSUoSUmN6PFi1RUG2dB6x5BKUclLSHHpbT47xJO+nmYdJnGTqlcZ2RU+r5x9MBxl+KA+CDBx306VYbpBMJjoV71eA/+0ovGjN65pjdllVeOR6AIrWw9vc8eeLBa06orA5+tPFEx1UkreqcAfdvGPCeqIJ1eT4VCcU4mteh5VxSAk+PjrydlUsmerZJG20XoXS0gXg7UOzTgFsn2wozNGsd2/ZIDpi/zSAPsPG/I7hA6SQJR+1+7bIEJ1ck4WSP06uTx/fz45Y481lOtRu3C7XW3Wn5XbyYR+JlJh38o38+mSjh9UUVzuclXmr9uwMrxOC1qxdRR9W4Lhew2P8fSXTxcfJ1cTRZs9Tk0LH80BDYOCIn1oUWgUgfmEsCHdrXuRS827zNSeR0XA6qrIVkhreySxMcgOROprCSAyjOxQ0kXxNIWBhtZ/ex1PHYC4VtEyK4/4Z0NBT4SANiRlUC2CsuXqPnl/MabaKt3pSSO3r0C0VQnV6mkJYbunyna3O7ekszVC4VbHr2zxqJmAtQtV8w6urfpVu3jVvmgCe3rimgn22NAF72dAg4Zoe+KFrp3I2yYh2iFo6EX6FsAIEMALSQkqb1bXB/fl2CXKKAWvgdRAq7bnXowxuWtZRaBhbELT4taeFGStm9p9m6dtxfkJWsWe5NyNe5wSuVSdTNGHZ3IMxmI4IMXfTDkcdFIaW+wak1qNRN1uhAUt2pvq0WAp4h3I3A5VRXVrWqDFhXNtN3TPeLAx46NS+D0wkrGl677bE+r8+jbJKw9CSVbSvs77rx8/Mkn6V5OT058ml+zk69n5Fbs4+fLFDOmq2UatshGetSylYfoDHKgv0Z2tddeLFy9OgBKWWP+GJpCRY5KVMNcFvoH8TtKXL2dDjE2k8va6/iYEo49GyPJqcxv8P5En3r4n/KaOUcQKFeq+OjBttXTDa236TuuicqvMRlIRewVe2sOOnfvTzHrdEX3ZAyY8VPV+5oqqCjjSh4qkdeFTwIa+eFR7oQLvhgB3rjfXFei7jH+6cKJ56XJu7ilY22xVcqKlFtWbCBS57MuwVMeFbt3hdBuOnNZNNa7lnQHlEUedxIdyulQrytzuFM+rIVLkvmL64zC6eqghiRt5jc5daQWNueB63JtKoJu+lCnDPh4+yitWG5MKdOOXPlKB9SzjcU8VyuawZm8aqL7utkjBri6oypWWoQo3syDdwhjsBhduCSh1MyR0r4un141hdPfFgHG81VzvO7dTdc8FAwogATh6AjIiXKgPaCAlDf4ceuQNZnlOrRvM9MERoMH6vdmSYkM465AuoTb4rH23+al7zvWHTV62PmvS/IBCF68WWns416tSvPHt9tU5yUvVu5c69Y7GqDNG0RHBeFejbwqp77ZMcjk5Ofs0oboxeDLX+34+v7o6V7oIP0x0LD8Sxy4/vBvJ1MDow9XoZ7p2KjzmbMC4c4a1APQdkwq07eHbEQgMhsFg/IgfloA+0K0kweoP+GX6qwRIW6iWtkx/Qp9LsvoDPmS4RX5HkEo7BM5Zf5aOqdiHDvrL7xZuXIKqg+rPEFqBchZEQV5i4JDKOWdVnERqWV4GlGm1apC3YUlhmxdLccz09/fu7++98A7YTHgpL0fqtMWIlNno/1Jn8JS26ql7tUrg0M3C9PZOrpZt5qNtAkKsK7eV73Ft3b+3L7hiu7P7VdXpVkt/RzjVSclq/gbA+oMpVlBSqXhUnOq7CXWd4nfsoi40Qv0P1DqzjlreIlshqVVpHoC7o4r2dclSkGTpgj7cRZ+5xDEzOFAZoI1m6hqZ0vXamVCfc3SsTlqsqBt0zUo8Igb1bUpXDQZpl6bgyLly/EBKSSvPH+GlNB5WYFg5pyAPribs6uTdx4k2/PCmNtjGv4FJfHn+6eTyd/bz5PehCdJT01CVsdFv85IkDTv/fDX5MLkc2F+8qEPzuwYsjF9t+9M16MsgXbnO+ecvk8srXPEXkw349eTj18kX93+H6n+DzSGra7cwdwyK6+atgunQhpra65bC2K6DHr/EBldaUUopRDPf3P3UVChPSUXFJQ35msicLmUMOvVmvWMaFWjPQPLuMTHlz5J5LfUaWqjHms92SlIp0DVvYLnizNzSlJ4cubTkke0yYSMFROH68LqOwncdjen6eaFR7bX1R5u311H2wbsRez1bsV3ydcOl1ad0LNPYtTREQQu+ibkLUtpl/UJ9Fg2sO+NVtsLLe3sxfoYIpaLvk5ns+0sE1ncal/lAeJKNoL+sKShvr996J8WiWgLLX1CLO7C64Zch/EC1u47U6uhx3WRxyMX42kSjHUMRSHXKwdZk2zvZ/j5e2Y4XTu05WqITv6Ykr/niSPkRDnynwMOK+yCBp2t86UnAppbEotdyAbC3/h8yc0E6'}}
SOURCE = LOCAL/'source'
SOURCE.mkdir(exist_ok=True)
for name, record in EMBEDDED.items():
    data = zlib.decompress(base64.b64decode(record['zlib_base64']))
    assert hashlib.sha256(data).hexdigest() == record['sha256'], name
    (SOURCE/name).write_bytes(data)
sys.path.insert(0, str(SOURCE))
os.environ['PYTHONPATH'] = str(SOURCE)
os.environ['MAX_JOBS'] = '2'
major, minor = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
from bootstrap_teachers import clone
from teachers import DMD_CODE
from extension_compat import prepare_build
from colab_dmd3cpp_full_train import freeze, publish
clone('https://github.com/Sharpiless/DMD3Cpp.git', DMD_CODE, LOCAL/'third_party'/'DMD3Cpp')
build = prepare_build(LOCAL/'third_party')
subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=build, check=True)
sys.path.insert(0, str(build))
freeze(OUT/'source_manifest.json', {name:r['sha256'] for name,r in EMBEDDED.items()})
for name in EMBEDDED:
    publish(SOURCE/name, OUT/'source'/name)
publish(CONFIG, OUT/'source'/'export_config.json')
print('Pinned metric-only source and BpOps ready. Full generation starts in the next cell.')

## Generate and commit all training samples

The generator verifies the exact 85,898-sample split, paired sparse inputs and
train/validation drive separation. HTTP range reads fetch the required PNG members
from official archives; unrelated raw sensors/frames and validation/test data are omitted.
Every sample is decoded, hashed, resized with the production adapter, and inferred by
`MetricTeacher`. A real-input GPU smoke must pass. Each teacher tensor is checked for
shape, float32 dtype, finite values, confidence bounds and usable KD support.

Each shard is ZIP/CRC/NPY audited locally, copied to Drive, then reopened and SHA-256
verified before its commit JSON is written. `status.json` tracks committed coverage.
A failed or interrupted transfer never counts as complete. Rerun cells 1–6 after a
runtime reset; committed shards are verified and skipped. Do not change the recipe.

The generation cell installs a separately audited confidence boundary guard. It
preserves depth and all already-valid tensors exactly, and clips only finite float32
confidence excursions within 1e-6 of [0,1]. Larger excursions or nonfinite values still
fail with the sample ID and measured range. The policy, code and correction events
are saved under `repairs/confidence_roundoff_v1/` and referenced in the final audit.
This keeps the original provider/exporter files and base recipe intact, allowing
already-verified shards to be reused without rewriting their data or provenance.

Create a file named `STOP` in the output folder to stop at the next shard boundary.
Remove that file yourself when you intend to resume. Interrupting the running cell
also stops work; the uncommitted shard will be regenerated.

In [ ]:
# @title 6. Generate DMD3C++ for the FULL official train split and save to Drive
import base64, hashlib, importlib, zlib
# A separately audited numerical correction; original provider/exporter files
# and base recipe remain unchanged so valid committed shards can be resumed.
_repair_bytes = zlib.decompress(base64.b64decode('eNqVWG1v47gR/u5fwfqLJJyj+GVjx95zgVyaAxbo3hW7e1+aBgItUTZ7kiiQVBIjzX/vDEnJlO1kb4PAkEVyOJx5nmeGHg6HN03GNctGZCOaKmMZyQtB9WxKUlHlPGNVyohkNeWS5EISvWNESL7lFS3IrSjohrDnWkgdDwbfdlwR+KdENXVdsJJVGmZVTckkT+GpFgVP9yNSCQ2TJHuSsDURubG6oQp3SnmNOwxqKR5he9iVF0zF5KaQjGZ78kgLnvnOaVYpIWFXict1Iys4xGav2QX4e4EP8eAz0+AByVitd+hhxR7BcikynnOWxeQz1eAieMikNLaqDJyELSp0MKMajqR5UZCc8iIeDIfDQS5FSWqqdwXfEF5iCMi/4OvAPe+owqH263+VqNpnodonzUvWPkOY6j2h4FwNofz9n3dfbn67vSNrMmEX88EvN1/vkq+///Hl9u4rvHsZEPgLNKPpjkkV1/tgRYIlvR4vJtPJcpZurj7M6SZdLGazTTqmjC4+TJd0Mb3O5yyjy1mWppvp9Xz+4Wo+oRs6yafXy3EwsmZTzGuSldksreskb4oi0ZLyym0zW47Hi+l8Tq/SWTabzZYfMjA32cyWS5ql4+n4apGPx/DI8vliMR6Pp4xOrpfTyXixWUwWU9jmdTAYZCy3oEtKk57kkNSQSkn3akQUz6KVcQpiflvwmoiq2BOXmBaqEq3wagtQTBupuKhgaS2ZYvIRX1vMWJsxJg8N8rx9o3YUMPe3NQmnIzK7gg+I4TwCFLYzMr23M6o6bjeFUcQxvOHK+uO8jmJaFKFzG/8gdADtL02F6b5DhIV58KmyXtmzE5dJh2ayYQBedsrBFXmBkLwGkTGeAhLsnveTB/OmEE8jsuPbHYwYR8M0LnkVRtHo8J0+w/c2BrCC/EwuDoCD3Y2Bv5MJ+Yl0779znI5BnsuGTEY1jNMrAp6sX2DDVTyZbl9HBDxZv+Bm9sVHEnSbnPnLA5t7F38DHSr3LWd/BlZ03r6C7T1E0YWNZW3EYJGG2PBKh2GYwsnHEfkfgSc4bhTFqim92GB+zQrv8EZhSIvP30TFBva93afLSJyKeh9aU4CRFLAbpiMyBnCNiGj0ul0BqYsGnun2/Yi8BBA2IBx8jpCUbWATiGNiEQKjJuW9Ufp8GMXojt6IKiySILig/knNn1mhYL45LtjTomCSgjl4d4iqYy2vlEaMwzHqRidSCH1g6Sc7aDT9SdK6BlSjntbNpuAK5LcsG003BRuBHVBkBh+VhsSiYkIAUfhZhVsfqOrU0qmd/+4tqUIhtYWJSTPf6LUZN3Luln/99O+7w3ADGq/aIapFCaKEwg06tKPTqzl4jL5yvR+0ENENFLoQzURGQQ7y8S5hgj8q8M7EvlUAG0zkDxwg1Q6wSjQScoohBmhhgQk70U8SrI1JEsU1FL9KmwWwtAEaAhkqWrKV8zz07FziQGR4iU+QBeJXl9dOHa0lOJU//P6xTAvg2gVIfcGZsoqtRb91qHmFddqm6LIt9h/deUEpVEl1unNR8IDWRsHHns9X7z0Ic5Jx+R0pDj4bTfiH5I/MAFU2lfHV+kYU0xqKiCIpA1Q7YXYMwOKCU+2JnbM1Om5S0/raJcqOmy4IMxQoSGRJgWKgCQEmA8urxcPFgdEXTvIuWsm7eJwEx6Q+z1gwi31V4tJvwQATbG5PbNhznMx2GDocLDpZaVqrpKnSHa22DFXrm2yA4gFIlcRXiZXF0ynHlkpXSRIhk64HS6zIY3hMCk+Pb45p20czyzzFyF4UVlAEln2EXAE8S1rxnCn9F1rV023YM1cIBwyOzIxDCtOFAmYKOkVKdzC3hwYPStSkXldKDpF49UpIC3IPx5cuLyq49HXedD0izxNAw7GBuPwTkW91Qa1tLozvifjTfLVY7GKx9ldfBvb4JnxBy652bmzsqDAybMEpMeAzU2E3ATv1RLNnDbUU9cNae5+Fdy6wZ+4djjDQrOcgex/hdd4onKoAikBdmFbQ1OTQUbBV/tgVHQ+6o/5B28rhx9WMY69rjXmFoDviyDnlpMeqgan+WwCb1vKg0vbu8c1+jaErzQAtQAx/S2usJZxtK3yxwBHsWlz1CXub597+eLcB+OD6Q7h7M7p0oM1VD97n0nID5rAv9bJiYmnI465knXEXMLdnG4kuGRSvmn/l5D/q5O2Jc9YjquEqQJ6gFYB+DxSkqZGcTsKNN2ZgK81Ez/taYoPo2+26zW0DvD89+ioYeS5js92onUc0r8Gz02xdczqROFhAjt/Hje07sAczfmDnZgdCxYoc+hRagpp5Bc8J0Ppkq/6Co/mgFY/MNMrfvaCZ9femU33opd9aOAtIQ1Ecvg/6+X8gfuL7eTeC3pbVI3XstD44t4dXGvyNOjZ5Eubt0lOx6JxdmCxMDpqKPycKvlVQEdA0YjPGj7C/8InrHQl7AmRsKeN8EUSxgH44DGgQYQNbiO3qpHeHl7H54cT6nTVlrUJjZQSwhNtAAs3z+ldaKCDSTyT4jw9s34iBaHg6JlScq32VhmYSSqwIjwJg+ZH7BGmrUXd/wauqBccQwDF8eON6lwemPVjfv9ioDs/ec4YPq3iJV8Zzs7r7jpv18NZO9pazPtjoX3/Ax+Bd5lrQWw4e8S9+U9ROIX2yNLH3pCxJzhC1LxRWtdakL6oHWaD257wEkeb8CNN86ylCKsqSa6CuFE/4aUBsHky7uz7ay6z2eW06Wnh5H2TYMieWikB+IIwSxSNgHvXb78O7gR+Tclv28TfFrnrhpYW6ToB1Xbu7OuWigDuEh3bLrQR/pTtucHq08/KMMQA185rC08Qair/0jhLgHtAMei0QVh8Ijxa9q0pMwR+h+HMYQSU87rDb5Uf99ZFKrvxKMzh3o4dqlpgTJsDhkz7eC0vUKbX9fujtgA/MyPaxK5JDY972vEnX87qGOGmbbdfevx6S0SJWUQiL6aZ6Km7QZkW8BeMJAd/HrmXBGQb8CDnPLj9PUEs9XPR+0causE/5fpPacvrM1gOvG/nUscBN9Juyfn8CN532NrjGn4/jniAGv3g/tZufZiR75KJRcIexEUbj7e/qbULjM9rY72j+DzQb7Yc='))
assert hashlib.sha256(_repair_bytes).hexdigest() == '46dd8fc83689a76971776780891d1a475e946790335701087d082da627ce2472'
(SOURCE/'dmd3cpp_confidence_repair.py').write_bytes(_repair_bytes)
importlib.invalidate_caches()
import dmd3cpp_confidence_repair
dmd3cpp_confidence_repair.install(OUT)
from colab_dmd3cpp_full_train import generate
generate(cfg)

## Consolidate and verify the full training ZIP

After all teacher shards pass audit, the next cell assembles
`kitti_train_full_85898.zip` on Colab SSD from the verified input shards. It contains
the original-resolution RGB, uint16 sparse/GT PNGs, camera calibration and full index.
The complete ZIP is CRC checked, copied to Drive, and hash verified there.
`COMPLETE.json` appears only after both full-train inputs and all teacher shards are verified.

Teacher ZIPs unpack to `metric/<hash-prefix>/<sample-id>.npy`, two float32 channels
`[D_cm, C_cm]`. Place `metric_metadata/{recipe.json,records.sqlite3}` into `metric/`
after extracting shards. Input files were indexed with fixed mtime 1600000000 seconds;
restore that mtime after extracting the input ZIP or rebuild input metadata before using
the production adapter. This is the metric portion only; the existing two-teacher
student launcher still requires a separate relative teacher and must not be run unchanged.

In [ ]:
# @title 7. Save the single full-train ZIP and final teacher manifest to Drive
from colab_dmd3cpp_full_train import finalize
finalize(cfg)

In [ ]:
# @title 8. Display verified completion status
from colab_dmd3cpp_full_train import read_json
complete = read_json(OUT/'COMPLETE.json')
assert complete['passed'] and complete['audit']['train_samples'] == 85898
assert complete['audit']['val_teachers'] == complete['audit']['test_teachers'] == 0
print(json.dumps(complete['audit'], indent=2))
print('Full KITTI training ZIP:', OUT/complete['full_train_zip']['file'])
print('Teacher dataset:', OUT/'teachers')
print('Verified completion record:', OUT/'COMPLETE.json')